# OpenNeuro 听觉 EEG：标准音与目标音识别

这是与 `01_sleep_edf_snn.ipynb` 分开的第二个小实验：原 Notebook 学习**睡眠分期**，本 Notebook 学习**听觉事件相关 EEG**。两者都使用 EEG，但任务、标签和数据格式不同，分开能更清楚地比较。

本实验使用 OpenNeuro 的公开数据集 [`ds006480`](https://openneuro.org/datasets/ds006480)，尝试从声音出现前后短短一段 EEG 中，区分标准音（standard tone）和目标音（target tone）。运行代码前先读第 1 节：单个受试者的原始 EEG 文件约 **1 GB**，Notebook 不会在你没有主动同意时自动下载。

**这是学习演示，不是临床工具或可靠的跨人群结论。** 我们只分析一位受试者的一种听觉条件；样本少、事件顺序和个体差异都会影响结果。

## 1. 先认识数据与下载大小

已核实的 OpenNeuro 数据集版本为 `ds006480` snapshot `1.0.1`。数据描述为年轻成人静息态与听觉 oddball EEG，包含普通对照和威胁条件；这里**只使用对照条件中的被动 oddball 任务**，不分析威胁条件。数据描述中的许可证为 CC0；学术报告仍应引用数据集作者与原研究，并遵守你的机构对公开人类 EEG 数据的要求。

数据 README 明确将事件码 `106` 定义为“被动听觉 oddball，对照条件，标准音”，`107` 定义为同一任务和条件中的“目标音”。`sub-01` EEG sidecar 记录 65 个 EEG 通道、1000 Hz 连续采样。我们仅下载 `sub-01` 的 EEG 文件（约 1.02 GB），并选其中 8 个通道用于演示；下载、校验和处理仍需要时间和磁盘空间。

在下一格把 `DOWNLOAD_EEG = True` **明确改为 `True` 之前**，确认电脑有至少 2.5 GB 可用磁盘空间并愿意下载约 1 GB 文件。若不想下载，可继续阅读说明，暂时不要运行下载单元。

In [ ]:
from pathlib import Path
import hashlib
import shutil
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "openneuro_ds006480" / "sub-01" / "eeg"
DATA_DIR.mkdir(parents=True, exist_ok=True)
EEG_FILE = DATA_DIR / "sub-01_task-oddball_eeg.set"
EVENTS_FILE = DATA_DIR / "sub-01_task-oddball_events.tsv"
EEG_URL = "https://s3.amazonaws.com/openneuro.org/ds006480/sub-01/eeg/sub-01_task-oddball_eeg.set"
EVENTS_URL = "https://raw.githubusercontent.com/OpenNeuroDatasets/ds006480/069fe5fe24ec3004e98304c5abb1442bb2d7420a/sub-01/eeg/sub-01_task-oddball_events.tsv"
EXPECTED_EEG_BYTES = 1_017_297_096
EXPECTED_EEG_SHA256 = "6006ad953c8be8cf9a74daf184e200fd9db708c78dfa73d423c5102f7aa7da8d"
DOWNLOAD_EEG = False  # 读完上面的说明、确认磁盘空间后，才改为 True。
SEED = 42
CHANNELS = [f"E{i}" for i in range(1, 9)]
DEVICE = torch.device("cpu")  # 小规模演示用 CPU，避免硬件兼容问题。

print("项目目录:", ROOT)
print("所选通道:", CHANNELS)
print("计算设备:", DEVICE)

### 为什么先确认再下载？

大型 EEG 原始文件不适合在不了解大小时悄悄下载。代码使用 macOS 自带的 `curl` 下载一个受试者文件和很小的事件表；如果中断，可再次运行继续下载。随后校验文件大小和 SHA-256，避免损坏文件或文件内容变化时继续分析。验证值对应数据集 snapshot `1.0.1` 中的 EEG 文件；如果数据提供方将来替换文件，校验会明确失败，而不是静默使用新数据。

数据文件保存在仓库的 `data/` 下，该目录已由 `.gitignore` 排除，不会随着普通 Git 提交上传。

In [ ]:
if not EEG_FILE.exists():
    if not DOWNLOAD_EEG:
        raise RuntimeError(
            "尚未同意下载约 1 GB 的 EEG 文件。确认磁盘空间后，将上一格 DOWNLOAD_EEG 改为 True，再运行本单元。"
        )
    free_bytes = shutil.disk_usage(ROOT).free
    if free_bytes < 2_500_000_000:
        raise OSError(
            f"可用磁盘空间不足：当前 {free_bytes / 1e9:.1f} GB，至少需要 2.5 GB。"
        )
    eeg_partial = EEG_FILE.with_suffix(EEG_FILE.suffix + ".partial")
    subprocess.run(
        ["curl", "--fail", "--location", "--retry", "3", "--continue-at", "-", "--output", str(eeg_partial), EEG_URL],
        check=True,
    )
    eeg_partial.replace(EEG_FILE)

if not EVENTS_FILE.exists():
    subprocess.run(
        ["curl", "--fail", "--location", "--silent", "--show-error", "--output", str(EVENTS_FILE), EVENTS_URL],
        check=True,
    )

actual_bytes = EEG_FILE.stat().st_size
if actual_bytes != EXPECTED_EEG_BYTES:
    raise ValueError(
        f"EEG 文件大小不符：预期 {EXPECTED_EEG_BYTES:,} 字节，实际 {actual_bytes:,} 字节。请检查下载是否完整。"
    )

digest = hashlib.sha256()
with EEG_FILE.open("rb") as eeg_stream:
    for chunk in iter(lambda: eeg_stream.read(1024 * 1024), b""):
        digest.update(chunk)
if digest.hexdigest() != EXPECTED_EEG_SHA256:
    raise ValueError("EEG 文件 SHA-256 与 snapshot 1.0.1 不一致；请勿继续分析。")

print(f"文件已下载并校验：{actual_bytes / 1e9:.2f} GB")
print("事件表:", EVENTS_FILE)

## 2. 读取 EEG 与事件表，切出听觉 epoch

BIDS 把连续 EEG 和事件标注保存在不同文件中。事件表里的 `onset` 是事件开始秒数、`value` 是数据 README 定义的事件码。我们只保留 106 和 107，再将秒数转换为采样点。

一段短时间窗称为 **epoch**。这里截取声音标记前 0.2 秒到后 0.8 秒，做 1–30 Hz 滤波和降采样。8 个通道而不是全部 65 个，可以减少学习演示的内存和计算负担；它们只是编号通道，不应在不知道电极坐标的情况下解释成特定脑区。

In [ ]:
event_table = pd.read_csv(EVENTS_FILE, sep="\t")
if not {"onset", "value"}.issubset(event_table.columns):
    raise ValueError(f"事件表缺少必要列，实际列名为：{event_table.columns.tolist()}")

class_codes = {"standard": 106, "target": 107}
selected = event_table.loc[
    event_table["value"].astype(str).str.strip().isin(["106", "107"])
].copy()
if selected.empty:
    raise ValueError("事件表中未找到 README 定义的标准音 106 或目标音 107。")

raw = mne.io.read_raw_eeglab(EEG_FILE, preload=False, verbose="ERROR")
missing_channels = sorted(set(CHANNELS) - set(raw.ch_names))
if missing_channels:
    raise ValueError(f"EEG 文件中缺少所选通道：{missing_channels}")
raw.pick(CHANNELS)
raw.load_data()
raw.filter(l_freq=1.0, h_freq=30.0, verbose="ERROR")
raw.resample(sfreq=250.0, verbose="ERROR")

sfreq = raw.info["sfreq"]
event_onsets = pd.to_numeric(selected["onset"], errors="raise").to_numpy()
event_values = pd.to_numeric(selected["value"], errors="raise").astype(int).to_numpy()
event_samples = np.rint(event_onsets * sfreq).astype(int) + raw.first_samp
events = np.column_stack(
    [event_samples, np.zeros(len(selected), dtype=int), event_values]
)

epochs = mne.Epochs(
    raw,
    events,
    event_id=class_codes,
    tmin=-0.2,
    tmax=0.8,
    baseline=(-0.2, 0.0),
    preload=True,
    reject_by_annotation=True,
    verbose="ERROR",
)
if len(epochs) == 0:
    raise ValueError("没有保留下来的 epoch；请检查 EEG、事件表和通道名称是否匹配。")

X = np.transpose(epochs.get_data(picks=CHANNELS), (0, 2, 1)).astype(np.float32)
y = (epochs.events[:, 2] == class_codes["target"]).astype(np.int64)
times = epochs.times
if not np.array_equal(np.unique(y), np.array([0, 1])):
    raise ValueError("清理后必须同时包含标准音和目标音，才能训练二分类器。")

print("EEG 数组 [试次, 时间点, 通道]:", X.shape)
print("采样率:", sfreq, "Hz；epoch 时间窗:", (times[0], times[-1]), "秒")
print("事件数 [标准音, 目标音]:", np.bincount(y, minlength=2))

In [ ]:
channel_to_plot = CHANNELS.index("E1")
standard_average = X[y == 0, :, channel_to_plot].mean(axis=0) * 1e6
target_average = X[y == 1, :, channel_to_plot].mean(axis=0) * 1e6

plt.figure(figsize=(9, 4))
plt.plot(times, standard_average, label="标准音")
plt.plot(times, target_average, label="目标音")
plt.axvline(0, color="black", linestyle="--", linewidth=1, label="声音事件")
plt.xlabel("相对事件时间（秒）")
plt.ylabel("平均电位（微伏，µV）")
plt.title("E1 通道：标准音与目标音的平均 EEG")
plt.legend()
plt.tight_layout()
plt.show()

### 怎样看上面的波形？

横轴 0 秒表示数据记录的声音事件码；纵轴是许多试次在同一通道上的平均电位，单位转换成了微伏。声音出现后脑电常会产生事件相关反应，但图上的起伏不能单独证明因果、听力状态或某种具体心理过程。这里只有一个编号电极，也不是完整的 ERP 科研分析。

## 3. 按时间先后划分训练集与测试集

同一段连续 EEG 中相邻试次可能非常相似。若随机打乱试次，训练和测试可能同时包含几乎相邻的信号，分数会显得过于乐观。这里按事件的时间顺序切分：前 80% 作为训练，最后 20% 作为测试。这样更适合检查“用较早试次训练，能否识别较后试次”，但仍然是**同一受试者**，不是跨人泛化评估。

先看两段中两种类别的数量。如果某类不在训练段或测试段，代码会停止并要求检查数据，而不会悄悄给出误导性的分数。

In [ ]:
split_at = int(len(X) * 0.8)
if split_at == 0 or split_at == len(X):
    raise ValueError("试次数太少，无法进行时间顺序划分。")

X_train, X_test = X[:split_at], X[split_at:]
y_train, y_test = y[:split_at], y[split_at:]
if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
    raise ValueError(
        "时间顺序切分后，训练集或测试集缺少一个类别。"
        "请不要随机混合相邻试次；可先检查所选受试者是否有足够目标音事件。"
    )

print("训练试次 [标准音, 目标音]:", np.bincount(y_train, minlength=2))
print("测试试次 [标准音, 目标音]:", np.bincount(y_test, minlength=2))

## 4. 传统 EEG 分类基线：SVM

作为对照，SVM（支持向量机）把每个 epoch 展平为“时间点 × 通道”的数字向量，再学习区分两种标签。它不模拟神经元发放脉冲，是常见的传统分类方法。

标准化器只在训练集拟合，然后应用于测试集。`class_weight="balanced"` 让较少出现的类别在训练时也有足够影响。此基线能回答一个重要问题：SNN 的结果是否至少与一个简单传统方法具有可比性？

In [ ]:
baseline = make_pipeline(
    StandardScaler(),
    SVC(class_weight="balanced", random_state=SEED),
)
baseline.fit(X_train.reshape(len(X_train), -1), y_train)
baseline_prediction = baseline.predict(X_test.reshape(len(X_test), -1))
print("SVM 基线完成；accuracy:", round(float(np.mean(baseline_prediction == y_test)), 3))

## 5. 把连续 EEG 转换为 SNN 能处理的脉冲

上一节的 SVM 直接读取一串连续数值。本实验的 SNN 则把每个时间点标准化后的正向和负向振幅分别作为两路输入，再缩放为 0–1 之间的发放概率，抽样成 0/1 事件（脉冲）。分成正、负两路是因为普通脉冲只能表示“发放/不发放”，而 EEG 振幅有正有负。SNN 看到的形状是 `[时间步, 试次, 通道]`，这里每个原始通道会对应正、负两路，时间步来自降采样后的真实 epoch 顺序。

每个训练通道的均值和标准差只用训练试次估计，再应用于测试试次，避免泄漏。请注意：这是对数字 EEG 的**软件脉冲编码**；它不是设备直接测得的神经元脉冲，也不等于真实的生物神经活动。随机编码还会引入波动，所以种子和编码方式都是实验参数。

In [ ]:
channel_mean = X_train.mean(axis=(0, 1), keepdims=True)
channel_std = X_train.std(axis=(0, 1), keepdims=True)
channel_std = np.where(channel_std < 1e-12, 1.0, channel_std)
X_train_z = ((X_train - channel_mean) / channel_std).astype(np.float32)
X_test_z = ((X_test - channel_mean) / channel_std).astype(np.float32)

def encode_eeg_as_spikes(values: np.ndarray, seed: int) -> torch.Tensor:
    """Encode [trial, time, channel] values as [time, trial, channel] spikes."""
    signal = torch.as_tensor(values, dtype=torch.float32, device=DEVICE)
    positive = signal.clamp(min=0.0)
    negative = (-signal).clamp(min=0.0)
    probabilities = torch.cat([positive, negative], dim=-1).div(3.0).clamp(0.0, 1.0)
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    random_values = torch.rand(
        probabilities.shape,
        dtype=probabilities.dtype,
        device=DEVICE,
        generator=generator,
    )
    spikes = (random_values < probabilities).to(torch.float32)
    return spikes.permute(1, 0, 2).contiguous()

train_spikes = encode_eeg_as_spikes(X_train_z, SEED)
test_spikes = encode_eeg_as_spikes(X_test_z, SEED + 1)
print("脉冲张量 [时间步, 试次, 通道]:", tuple(train_spikes.shape))
print("训练集脉冲比例:", round(float(train_spikes.mean()), 3))

## 6. 训练 LIF 脉冲神经网络

LIF（leaky integrate-and-fire，漏积分发放）神经元会随时间累积输入，达到阈值时发放脉冲。隐藏层把输入脉冲变换为新的脉冲；输出层也以脉冲活动工作，最终累加各类别在整个时间窗的输出脉冲作为分类分数。

阈值发放本身不可直接求普通导数。snnTorch 使用 surrogate gradient（替代梯度）近似反向传播时的梯度，使模型可以用交叉熵和 Adam 训练。这里用类别权重缓解标准音、目标音数量不平衡；`EPOCHS` 是训练轮数。模型小、样本少，损失下降不代表测试泛化一定好。

In [ ]:
from snn_eeg.models import TwoLayerSNN

torch.manual_seed(SEED)
y_train_tensor = torch.as_tensor(y_train, dtype=torch.long, device=DEVICE)
class_counts = torch.bincount(y_train_tensor, minlength=2).float()
class_weights = class_counts.sum() / (2.0 * class_counts.clamp_min(1.0))

snn_model = TwoLayerSNN(
    n_inputs=2 * len(CHANNELS), n_hidden=32, n_classes=2
).to(DEVICE)
optimizer = torch.optim.Adam(snn_model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
EPOCHS = 20

for epoch in range(EPOCHS):
    snn_model.train()
    optimizer.zero_grad()
    output_counts, _ = snn_model(train_spikes)
    loss = loss_fn(output_counts, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"训练轮数 {epoch + 1:02d} / {EPOCHS}，loss={loss.item():.4f}")

## 7. 测试集评估：正确率、逐类表现与混淆矩阵

训练时从未使用最后一段测试试次的标签。现在分别评估 SVM 和 SNN。除了 accuracy，还要查看逐类 precision、recall、F1 和混淆矩阵；若目标音数量较少，只报 accuracy 可能会掩盖模型忽略了目标音的问题。

结果不同于或好于 SVM 都是可能的：本实验样本量小，脉冲编码带随机性，且单受试者的时间顺序划分会造成分布变化。分数不是不同模型普遍优劣的证明。

In [ ]:
snn_model.eval()
with torch.no_grad():
    snn_counts, activity = snn_model(test_spikes)
    snn_prediction = snn_counts.argmax(dim=1).cpu().numpy()

print("SVM：")
print(classification_report(
    y_test, baseline_prediction,
    labels=[0, 1], target_names=["标准音", "目标音"], zero_division=0,
))
print("SNN：")
print(classification_report(
    y_test, snn_prediction,
    labels=[0, 1], target_names=["标准音", "目标音"], zero_division=0,
))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for axis, prediction, title in zip(
    axes, [baseline_prediction, snn_prediction], ["SVM baseline", "LIF SNN"]
):
    ConfusionMatrixDisplay.from_predictions(
        y_test, prediction,
        labels=[0, 1], display_labels=["Standard", "Target"],
        cmap="Blues", colorbar=False, ax=axis,
    )
    axis.set_title(title)
plt.tight_layout()
plt.show()

## 8. 观察 SNN 的脉冲活动

模型除了预测类别，还返回隐藏层和输出层每个时间步的脉冲。下面报告测试集隐藏层脉冲数，并画出一个测试试次的隐藏神经元活动栅格：亮点表示某个计算时间步发放了脉冲。

这些数值能描述这个软件模型的活动模式；它们**不能直接换算成焦耳、耗电量或能效优势**。真实能耗比较还需要相同任务、相同精度和硬件条件下的设备级测量。

In [ ]:
hidden_spikes = activity["hidden_spikes"].cpu().numpy()
output_spikes = activity["output_spikes"].cpu().numpy()
print("测试集隐藏层平均发放率:", round(float(hidden_spikes.mean()), 4))
print("测试集隐藏层脉冲数:", int(hidden_spikes.sum()))

example_index = 0
plt.figure(figsize=(9, 4))
plt.imshow(
    hidden_spikes[:, example_index, :].T,
    aspect="auto", origin="lower", interpolation="nearest", cmap="Greys",
)
plt.xlabel("epoch 时间步")
plt.ylabel("隐藏层神经元")
plt.title("一个测试试次的隐藏层脉冲栅格")
plt.tight_layout()
plt.show()

## 9. 小结：如何解释这个实验

本次流程用公开 BIDS 数据中的标准音／目标音事件码，读取单人连续 EEG，做小规模预处理，随后比较 SVM 与时序 LIF SNN。SVM 读取展平后的数值，SNN 则读取按时间排列的随机二值脉冲，并让 LIF 状态随时间演化。这展示了两种**计算表示和模型机制**的差异，不意味着 SNN 在本任务上一定更准确、更省电或更符合生物神经活动。

下一步可以在多个受试者上重复分析、尝试更合理的电极空间选择、检查伪迹和预处理参数，并制定不泄漏时间信息的评估方案。若研究目标是跨人推广，应将整位受试者留作测试，不能用随机试次划分代替。

**数据来源和事件码：** [OpenNeuro ds006480](https://openneuro.org/datasets/ds006480)，snapshot `1.0.1`；[数据集 README（事件码定义）](https://github.com/OpenNeuroDatasets/ds006480/blob/069fe5fe24ec3004e98304c5abb1442bb2d7420a/README#L1-L49)；[数据说明与许可证](https://github.com/OpenNeuroDatasets/ds006480/blob/069fe5fe24ec3004e98304c5abb1442bb2d7420a/dataset_description.json)。原研究预印本 DOI：[10.1101/2025.10.02.680040](https://doi.org/10.1101/2025.10.02.680040)。